# Reproducing Zandvakili and Kohn (2015) on CRCNS v1v2-1

Paper: A. Zandvakili and A. Kohn, "Coordinated Neuronal Activity Enhances Corticocortical
Communication", Neuron 87, 827-839 (2015), doi:10.1016/j.neuron.2015.07.026.

Dataset: CRCNS v1v2-1, "Simultaneous V1-V2 neuronal population recordings in anesthetized
macaque monkeys" (Zandvakili and Kohn 2019), doi:10.6080/K0B27SHN, read from the public S3
bucket `crcnsarchive` at `v1v2-1/live/v1-v2_gratings.zip`, opened in place.

Version: CRCNS data sets are not versioned, so the version is recorded as `unversioned`
(docs/archives.md, CRCNS). The bucket holds one copy of the data, under `live/` (uploaded
2026-02-11), with data description version 0.5 (Feb 15, 2019); there is no other version to
choose, and the paper cites none. The files read are pinned by the MD5 sums of the data set's
`checksums.md5`: `v1-v2_gratings.zip` 12a15c5a63366ccd1fa2b4e495f1d66d,
`docs/crcns_v1v2-1_data_description.pdf` 72e436e8e0c006efbbe5eab4737c3c69,
`docs/sessions_description.pdf` 48917aea682f869c7aee45f170516c7d. The first cell below checks
the listed sum of the zip against these.

Claims: `claims/zandvakili-2015.yaml`, written from the version of record with its supplement.
The data set holds 5 sessions from 3 animals of the paper's 7; the paper's counts describe the
larger sample, so fractions, medians and comparisons are what is tested.

| claim | here |
|---|---|
| fig4c-broad-ccg-peaks | tested |
| fig4c-narrow-peak-sites | tested |
| fig4-narrow-peak-depth | blocked, data_missing (no V2 depths) |
| fig5a-rf-alignment | blocked, metadata_insufficient (RF positions in pixels, no scale) |
| fig5-narrow-pairs-rare | tested |
| fig5b-v1v2-latency | tested |
| fig5b-intra-areal-zero-lag | tested |
| figs3-efficacy | tested |
| fig6a-coordination-before-v2 | tested |
| fig7a-threefold | tested |
| fig7a-above-shuffle | tested |
| fig6b-not-after-v2 | tested |
| fig6-slow-both-sides | tested |
| fig7d-time-course | tested |
| fig7b-nonmiddle-no-coordination | blocked, data_missing (no V2 depths or penetrations) |
| fig7e-nonmiddle-time-course | blocked, data_missing (as fig7b) |
| fig7c-offset-rf-no-coordination | blocked, data_missing (no depths; RF scale missing) |
| figs2a-v2-single-units | blocked, data_missing (no unit SNR or waveforms) |
| figs2b-v1-single-units | blocked, data_missing (no unit SNR or waveforms) |
| figs4-partner-not-needed | tested |
| figs5-middle-layer-relay | blocked, data_missing (no V2 depths) |
| fig1-ffn-propagation, fig3-divergence-correlation, figs1-higher-order, fig8-architecture | blocked, not_runnable (simulations, data: elsewhere, no code; not run in this attempt's time) |
| rf-sizes | blocked, metadata_insufficient (RF fits in pixels, no scale) |

Shared methods are in the helper `v1v2.py`, whose docstring says where each parameter comes
from. Random seeds are fixed; each claim's section draws from its own generator.

In [ ]:
import time, resource
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import fsspec
import v1v2 as h

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
T0 = time.time()
z = h.open_zip()
md5 = fsspec.open("https://crcnsarchive.s3-us-west-2.amazonaws.com/v1v2-1/live/checksums.md5").open().read().decode()
print(md5)
assert "12a15c5a63366ccd1fa2b4e495f1d66d  v1-v2_gratings.zip" in md5
for i in z.infolist():
    print(f"{i.filename:60s} {i.file_size:>10d}")

## Load and compute

Each session is loaded, its CCGs computed for all pairs of kept units (both jitter windows),
and the V1 state histograms around every V2 spike computed (helper `session_ccgs`,
`session_divergence`). The session's spikes are then released, so that one session is in
memory at a time. This is the long cell (several minutes).

In [ ]:
L = 2 * h.MAXLAG + 1
LAGS = np.arange(-h.MAXLAG, h.MAXLAG + 1)
pairs, sites, sess_info, div, sig_ccgs, clock, dup = [], [], {}, {}, [], [], []

def decode_acc(X, y, rng):
    """Nearest class mean on z-scored counts, two-fold cross-validation (a standard choice)."""
    X = (X - X.mean(0)) / (X.std(0) + 1e-9)
    idx = rng.permutation(len(y)); half = np.zeros(len(y), bool); half[idx[: len(y) // 2]] = True
    acc = []
    for tr in (half, ~half):
        cls = np.unique(y)
        mu = np.stack([X[tr & (y == c)].mean(0) for c in cls])
        d = ((X[~tr][:, None, :] - mu[None]) ** 2).sum(-1)
        acc.append(np.mean(cls[d.argmin(1)] == y[~tr]))
    return float(np.mean(acc))

for si, name in enumerate(h.SESSIONS):
    t = time.time()
    s = h.load_session(z, name)
    k1, k2 = h.keep_units(s, "V1"), h.keep_units(s, "V2")
    n_tr = len(s["stim"])
    sess_info[name] = dict(animal=name[:3], n_rows=s["n_rows"], n_driven=n_tr,
                           stim_counts=np.bincount(s["stim"])[1:].tolist(),
                           v1_units=s["V1"]["n_units"], v2_units=s["V2"]["n_units"],
                           v1_kept=len(k1), v2_kept=len(k2),
                           v1_rate=s["V1"]["rate"], v2_rate=s["V2"]["rate"],
                           v1_codes=s["V1"]["codes"], v2_codes=s["V2"]["codes"],
                           v1_rf=s["V1"]["rf"], v2_rf=s["V2"]["rf"],
                           trial_id_increasing=bool(np.all(np.diff(s["trial_id"]) > 0)),
                           row_alternates=bool(np.all(s["row_stim"][1::2] == 0) and np.all(s["row_stim"][0::2] > 0)))
    # clock check: decode orientation from V1 counts in windows around stimulus onset
    rng = np.random.default_rng(100 + si)
    u, tr, tt = s["V1"]["events"]; m = np.isin(u, k1)
    bu, btr, btt = s["V1"]["blank_events"]; bm = np.isin(bu, k1)
    umap = np.full(u.max() + 1, -1); umap[k1] = np.arange(len(k1))
    for lo, hi, label in [(-100, 0, "last 100 ms of preceding blank"), (0, 30, "0-30 ms"), (30, 60, "30-60 ms"),
                          (60, 160, "60-160 ms"), (1180, 1280, "1180-1280 ms")]:
        X = np.zeros((n_tr, len(k1)))
        if lo < 0:
            # blank row k precedes driven trial k + 1 (rows alternate driven, blank)
            mm = bm & (btt >= 1500 + lo) & (btr < n_tr - 1)
            np.add.at(X, (btr[mm] + 1, umap[bu[mm]]), 1)
            keep = np.arange(1, n_tr)
        else:
            mm = m & (tt >= lo) & (tt < hi)
            np.add.at(X, (tr[mm], umap[u[mm]]), 1)
            keep = np.arange(n_tr)
        acc = decode_acc(X[keep], s["stim"][keep], rng)
        # shifted labels: the same counts against the next trial's stimulus
        acc_shift = decode_acc(X[keep][:-1], s["stim"][keep][1:], rng)
        clock.append(dict(session=name, window=label, accuracy=acc, shifted_labels=acc_shift))
    # population PSTH onset (all orientations), for the latency check
    psth = np.bincount(tt[m], minlength=h.T_MS) / (n_tr * len(k1)) * 1000
    sess_info[name]["psth"] = psth
    # mean V1 population response per stimulus, 50-1280 ms, for the orientation check
    resp = np.zeros((8, len(k1)))
    mm = m & (tt >= 50)
    np.add.at(resp, (s["stim"][tr[mm]] - 1, umap[u[mm]]), 1)
    sess_info[name]["resp"] = resp / (np.bincount(s["stim"])[1:, None])

    cc = h.session_ccgs(s)
    units = cc["units"]
    area = np.array([a for a, _ in units]); uidx = np.array([i for _, i in units])
    elec = np.array([s[a]["codes"][i, 0] for a, i in units])
    nsp = cc["n_spikes"]
    iu, ju = np.triu_indices(len(units), 1)
    q = (h.T_MS - np.abs(LAGS)) / h.T_MS
    norm = q[None, :] * nsp[iu][:, None]
    raw = cc["raw"][iu, ju] / norm
    sc = (cc["raw"][iu, ju] - cc["shuffle"][iu, ju]) / norm
    jc = {w: (cc["raw"][iu, ju] - cc["jitter"][w][iu, ju]) / norm for w in cc["jitter"]}
    # duplicate check: fraction of the sparser unit's spikes coincident at lag 0
    zero_frac = cc["raw"][iu, ju, h.MAXLAG] / np.minimum(nsp[iu], nsp[ju])
    for k in np.where(zero_frac > 0.5)[0]:
        dup.append(dict(session=name, a=units[iu[k]], b=units[ju[k]], frac=zero_frac[k]))
    del cc
    ptype = np.char.add(area[iu], area[ju])
    df = pd.DataFrame(dict(session=name, animal=name[:3], type=ptype, i=uidx[iu], j=uidx[ju],
                           elec_i=elec[iu], elec_j=elec[ju], n_i=nsp[iu], n_j=nsp[ju],
                           zero_frac=zero_frac))
    for w in jc:
        sm = h.gauss_smooth(jc[w])
        sig, peak, lag, sd = h.significance(sm)
        df[f"sig{w}"] = sig; df[f"peak{w}"] = peak; df[f"lag{w}"] = lag; df[f"sd{w}"] = sd
        if w == 10:
            ks = np.where(sig)[0]
            li = lag[ks] + h.MAXLAG
            df.loc[ks, "width_s"] = [h.half_width(sm[k], l) for k, l in zip(ks, li)]
            df.loc[ks, "width_u"] = [h.half_width(jc[w][k], l) for k, l in zip(ks, li)]
            # variant: peak lag refined below the 1 ms bin by a parabola through the three
            # bins at the maximum of the smoothed CCG (not the paper's method, which had 0.1 ms bins)
            y0, ym, yp = sm[ks, li], sm[ks, li - 1], sm[ks, li + 1]
            df.loc[ks, "lag_interp"] = lag[ks] + 0.5 * (ym - yp) / (ym - 2 * y0 + yp)
            for nm, arr in [("eff_jc", jc[w]), ("eff_raw", raw), ("eff_sc", sc)]:
                df.loc[ks, nm] = [arr[k, l - 2:l + 3].sum() * 100 for k, l in zip(ks, li)]
            for k in ks:
                sig_ccgs.append((ptype[k], sm[k]))
    pairs.append(df)
    # site averages: V1-V2 pairs grouped by V2 electrode
    v12 = np.where(ptype == "V1V2")[0]
    for e in np.unique(elec[ju][v12]):
        kk = v12[elec[ju][v12] == e]
        row = dict(session=name, animal=name[:3], electrode=int(e), n_pairs=len(kk),
                   n_v2_units=len(np.unique(uidx[ju][kk])), sc=sc[kk].mean(0))
        for w in jc:
            sig, peak, lag, sd = h.significance(h.gauss_smooth(jc[w][kk].mean(0)))
            row[f"sig{w}"] = bool(sig); row[f"peak{w}"] = float(peak); row[f"lag{w}"] = int(lag); row[f"sd{w}"] = float(sd)
        sites.append(row)
    del raw, sc, jc
    div[name] = h.session_divergence(s, seed=1000 + si)
    div[name]["v2_elec"] = s["V2"]["codes"][div[name]["v2_units"], 0]
    del s
    print(f"{name}: {time.time() - t:.0f} s, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
pairs = pd.concat(pairs, ignore_index=True)
sites = pd.DataFrame(sites)
clock = pd.DataFrame(clock)
print(f"total {time.time() - T0:.0f} s")

## Data check

In [ ]:
info = pd.DataFrame({k: {c: v[c] for c in ["animal", "n_rows", "n_driven", "v1_units", "v1_kept", "v2_units", "v2_kept",
                                           "trial_id_increasing", "row_alternates"]} for k, v in sess_info.items()}).T
print(info)
print("trials per orientation code 1..8:", {k: v["stim_counts"] for k, v in sess_info.items()})
print("animals:", sorted({v["animal"] for v in sess_info.values()}), " sessions:", len(sess_info))
n1 = sum(v["v1_units"] for v in sess_info.values()); n2 = sum(v["v2_units"] for v in sess_info.values())
print(f"units in the files: V1 {n1} (mean {n1 / 5:.1f} per session; descriptor says 112.2), V2 {n2} (mean {n2 / 5:.1f}; descriptor 29.4)")
for a in ["v1", "v2"]:
    r = np.concatenate([v[f"{a}_rate"] for v in sess_info.values()])
    print(f"{a.upper()} driven rate, sp/s: min {r.min():.2f}, median {np.median(r):.2f}, max {r.max():.2f}; below 0.5 sp/s: {(r < 0.5).sum()} removed")
for k, v in sess_info.items():
    print(k, "V2 electrodes (sites):", sorted(set(v["v2_codes"][:, 0].tolist())),
          " V1 electrodes:", len(set(v["v1_codes"][:, 0].tolist())), "of 96",
          " rfInfo shapes V1", v["v1_rf"].shape, "V2", v["v2_rf"].shape)
allrf = np.concatenate([v["v1_rf"][:, :, 0] for v in sess_info.values()])
print("V1 RF centre x range (pixels) %.0f to %.0f, y %.0f to %.0f; variance x median %.0f px^2; r2 median %.2f" % (
    np.nanmin(allrf[:, 2]), np.nanmax(allrf[:, 2]), np.nanmin(allrf[:, 3]), np.nanmax(allrf[:, 3]),
    np.nanmedian(allrf[:, 4]), np.nanmedian(allrf[:, 6])))
for k, v in sess_info.items():
    if v["v1_rf"].shape[2] == 2:
        dx = np.abs(v["v1_rf"][:, 2, 1] - v["v1_rf"][:, 2, 0])
        print(f"{k}: two eyes mapped; horizontal distance between the eyes' V1 RF centres, median {np.nanmedian(dx):.0f} px (range {np.nanmin(dx):.0f} to {np.nanmax(dx):.0f})")
    else:
        print(f"{k}: one eye mapped")
print("spike rasters hold only 0 and 1 (asserted when loading); times are 1 ms bins 0..1279 (driven) and 0..1499 (blank)")
print("\nClock check: orientation decoded from V1 counts (chance 0.125), true labels and labels of the next trial")
print(clock.pivot(index="window", columns="session", values="accuracy").round(3))
print(clock.pivot(index="window", columns="session", values="shifted_labels").round(3))
print("\nV1 population PSTH (sp/s per unit), first ms where it exceeds the 0-20 ms mean by 50%:")
for k, v in sess_info.items():
    p = np.convolve(v["psth"], np.ones(5) / 5, "same"); base = p[:20].mean()
    print(f"  {k}: baseline {base:.1f}, onset {int(np.argmax(p > 1.5 * base))} ms, peak {p.max():.1f} at {int(p.argmax())} ms")
print("\nOrientation codes: correlation of mean V1 population responses between stimulus codes")
for k, v in sess_info.items():
    c = np.corrcoef(v["resp"])
    print(f"  {k}: code 7 with 1: {c[6, 0]:.3f}, with 6: {c[6, 5]:.3f}, with 8: {c[6, 7]:.3f}; code 5 with 1: {c[4, 0]:.3f}")
print(f"\nPairs with more than half of the sparser unit's spikes coincident at lag 0 (duplicate check): {len(dup)}")
for d in dup:
    print("  ", d)
print("pairs:", pairs.groupby(["session", "type"]).size().unstack())
print("V2 sites (electrodes):", len(sites), "; per session", sites.groupby("session").size().to_dict())

Checks of the description against the data:

- 5 sessions in 3 animals (105, 106, 107), as the descriptor says. Each file holds 6400 rows,
  alternating a driven trial (1280 ms) and a blank (1500 ms); 400 driven trials per
  orientation code. Trial ids increase.
- The descriptor gives an average of 112.2 V1 units per session; the files hold the count
  printed above. V2: 29.4, as printed.
- The V2 electrodes are tetrodes, with unit electrode codes 1, 5, 9, ... (the first channel
  of each tetrode); rfInfo for V2 has one row per channel (28), so a V2 unit's RF is at the
  row of its electrode code. V1 rfInfo has 96 rows, one per array channel.
- RF positions and variances are in pixels. Nothing in the data set gives the pixel-to-degree
  scale or the position of the fixation point, and in four sessions the two eyes' RFs are
  mapped at the horizontal distances printed above (two gratings, Visual Stimuli). The paper gives 1024 x 768 pixels
  at 110 cm without the screen size.
- Clock: orientation is decoded at chance from the end of the preceding blank and in the
  first 30 ms, and far above chance from 60 ms, with chance accuracy when the labels are
  shifted by one trial; the V1 population PSTH rises 58 to 68 ms after the start of the
  driven row, a typical V1 latency. This is consistent with each driven row starting at stimulus onset and with
  the labels belonging to their own trial.
- Orientation codes: the sessions description says code 7 is 180 deg in session 105l001p16
  (135 deg elsewhere). The population responses to code 7 correlate with those to code 1
  (0 deg) more than with codes 6 and 8 in that session only, consistent with that note.
- No pair of units shares more than half of its spikes at zero lag, apart from those listed
  above (none listed means no duplicated unit); the five sessions have different unit counts,
  so no session is deposited twice.
- Not in the data set: V2 recording depth, penetration identity, unit SNR or waveforms,
  which the paper's layer groups and single-unit analyses need.

## fig4c-narrow-peak-sites

> A significant narrow peak in the site-average V1-V2 CCG occurs at a minority of V2 sites (46
> of 223). Expected fraction 0.206 +/- 0.021.

A site is a V2 electrode in a session (one recording depth per session; the files hold no
depth). Site-average jitter-corrected CCG over all V1-V2 pairs at that electrode, 10 ms
jitter (reading j-a), smoothed, 5 SD rule. The 25 ms window is shown as a variant of reading
j-b.

In [ ]:
print(sites[["session", "electrode", "n_pairs", "n_v2_units", "sig10", "peak10", "sd10", "lag10", "sig25"]].round(6).to_string())
for w in (10, 25):
    print(f"jitter {w} ms: {sites[f'sig{w}'].sum()} of {len(sites)} sites significant, fraction {sites[f'sig{w}'].mean():.3f}")
print("per animal (10 ms):", sites.groupby("animal")["sig10"].agg(["sum", "count"]).to_dict("index"))
ratio = sites.peak10 / sites.sd10
print("peak / flank SD at each site, sorted:", np.sort(ratio.values).round(1))
for thr in (4, 5, 6):
    print(f"  threshold {thr} SD: {(ratio > thr).sum()} of {len(sites)}")

**Result: not reproduced.** 16 of 31 V2 sites (0.516) have a significant narrow peak in the
site-average CCG with the 10 ms jitter, and 22 of 31 (0.710) with the 25 ms variant, against
0.206 +/- 0.021. The fraction is above the paper's in every animal (3 of 6, 5 of 11, 8 of 14).
The data set holds 31 sites against the paper's 223, and the paper says some penetrations
targeted middle-layer sites, so the fraction depends on which sessions were deposited.

## fig4c-broad-ccg-peaks

> V1-V2 CCGs frequently have broad peaks, tens to hundreds of milliseconds wide, evident in
> site-average CCGs at all laminar locations.

Site-average shuffle-corrected CCGs. Readings (a) mean over -50..50 ms above the flank mean
(75 to 125 ms), (b) mean over -50..50 ms above zero.

In [ ]:
SC = np.stack(sites.sc.values)
centre = np.abs(LAGS) <= 50; flank = (np.abs(LAGS) >= 75)
sites["broad_a"] = SC[:, centre].mean(1) > SC[:, flank].mean(1)
sites["broad_b"] = SC[:, centre].mean(1) > 0
sites["centre_mean"] = SC[:, centre].mean(1); sites["flank_mean"] = SC[:, flank].mean(1)
print(sites[["session", "electrode", "sig10", "centre_mean", "flank_mean", "broad_a", "broad_b"]].to_string())
for r in "ab":
    g = sites.groupby("sig10")[f"broad_{r}"].agg(["sum", "count"])
    print(f"reading {r}: {sites[f'broad_{r}'].sum()} of {len(sites)} sites; by narrow peak:", g.to_dict("index"))
avg = SC.mean(0)
pk = h.MAXLAG + int(np.argmax(avg[np.abs(LAGS) <= 10])) - 10
w_zero = h.half_width(avg, pk)
w_flank = h.half_width(avg - avg[flank].mean(), pk)
print(f"across-site average: peak {avg.max():.2e} at {LAGS[avg.argmax()]} ms, flank mean {avg[flank].mean():.2e}; "
      f"width at half height above zero {w_zero:.0f} ms (250 means it does not fall to half within +-125 ms), above the flank mean {w_flank:.0f} ms")
print("range of the plotted site-average CCGs: %.2e to %.2e" % (SC.min(), SC.max()))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
for k in range(len(SC)):
    ax[int(sites.sig10[k])].plot(LAGS, SC[k], lw=0.7)
for a, t_ in zip(ax, ["sites without a narrow peak", "sites with a narrow peak"]):
    a.set_title(t_); a.set_xlabel("lag (ms), V2 after V1 > 0"); a.set_ylabel("coinc. per V1 spike")
plt.tight_layout()

**Result: reproduced.** Under both readings all 31 site-average shuffle-corrected CCGs have a
broad peak: 16 of 16 sites with a narrow peak and 15 of 15 without. The across-site average
peaks at 2 ms and is 68 ms wide at half its height above the flank mean; it stays above half
its height above zero over the whole +-125 ms.

## fig4-narrow-peak-depth

Blocked, `data_missing`: the files hold no recording depth for any V2 site, and no layer
6/white matter reference.

## fig5-narrow-pairs-rare

> Only about 0.3% of all V1-V2 pairs have a significant narrow CCG peak (296 of 83,441).
> Expected 0.003 +/- 0.0003 or 0.00355 +/- 0.000355.

In [ ]:
v12 = pairs[pairs.type == "V1V2"]
for w in (10, 25):
    print(f"jitter {w} ms: {v12[f'sig{w}'].sum()} of {len(v12)} V1-V2 pairs, fraction {v12[f'sig{w}'].mean():.5f}")
print(v12.groupby("session")["sig10"].agg(["sum", "count", "mean"]))
print("by animal:", v12.groupby("animal")["sig10"].agg(["sum", "count", "mean"]).to_dict("index"))
r = v12.peak10 / v12.sd10
for thr in (4, 5, 6):
    print(f"threshold {thr} SD: fraction {(r > thr).mean():.5f}")

**Result: not reproduced.** 171 of 16,308 V1-V2 pairs (0.0105) have a significant narrow
peak with the 10 ms jitter (185, 0.0113, with 25 ms), against 0.003 or 0.00355. The session
fractions are 0.0028 to 0.018; only 106r002p70 (0.0028) is within tolerance. Narrow peaks are
rare here too, about 1% of pairs. The site fraction (fig4c-narrow-peak-sites) is 2.5 times the
paper's, consistent with sessions richer in coupled sites than the paper's sample.

## fig5b-v1v2-latency

> Narrow V1-V2 CCG peaks are offset from zero lag by a median of 2.8 ms, significantly
> different from zero, with a mean peak width of 3.71 ms. Expected 2.8 +/- 0.28 ms and
> 3.71 +/- 0.1 ms; test p < 0.001.

In [ ]:
sg = v12[v12.sig10]
lag = sg.lag10.values
print(f"n = {len(sg)} significant V1-V2 pairs; median offset {np.median(lag):.1f} ms, mean {lag.mean():.2f} ms")
print("offsets (ms), counts:", dict(zip(*np.unique(lag, return_counts=True))))
print("Wilcoxon signed-rank (zeros dropped):", st.wilcoxon(lag[lag != 0]))
npos, nneg = (lag > 0).sum(), (lag < 0).sum()
print(f"sign test: {npos} positive, {nneg} negative, p = {st.binomtest(npos, npos + nneg).pvalue:.2e}")
print(f"width, (a) smoothed: mean {sg.width_s.mean():.2f} +/- {sg.width_s.std() / np.sqrt(len(sg)):.2f} ms (SEM); "
      f"(b) unsmoothed: mean {sg.width_u.mean():.2f} +/- {sg.width_u.std() / np.sqrt(len(sg)):.2f} ms")
print("per session median offset:", sg.groupby("session").lag10.median().to_dict(), " n:", sg.groupby("session").size().to_dict())
print("per animal median offset:", sg.groupby("animal").lag10.median().to_dict())
print("pairs with offset above 0: %d, at 0: %d, below 0: %d" % (npos, (lag == 0).sum(), nneg))
print(f"variant, peak lag interpolated below 1 ms: median {sg.lag_interp.median():.2f} ms, mean {sg.lag_interp.mean():.2f} ms")
sg25 = v12[v12.sig25]
print(f"variant, 25 ms jitter: n = {len(sg25)}, median offset {np.median(sg25.lag25):.1f} ms")

**Result: partially reproduced.** Of 171 significant V1-V2 pairs, 151 peak after zero lag and
15 before (sign test p = 1.9e-29, Wilcoxon p = 7.1e-20), so the offset differs from zero with
p < 0.001 as the paper reports. The median offset is 2.0 ms, outside 2.8 +/- 0.28. The files'
1 ms bins put the peaks at whole milliseconds (55 pairs at 2 ms, 49 at 3 ms); a parabola through
the peak bins gives a median of 2.30 ms, also outside. The mean width is 3.89 +/- 0.07 ms on the
smoothed CCG and 3.09 ms on the unsmoothed one, both outside 3.71 +/- 0.1.

## fig5b-intra-areal-zero-lag

> Significant narrow peaks of V1-V1 and V2-V2 jitter-corrected CCGs have a median offset of
> 0.0 ms, not different from zero. Claim of absence; Wilcoxon signed-rank (a) or sign test
> (b), two-sided, p < 0.05.

The lag sign follows the order of the unit lists (claims file). The data put units of one
tetrode, and units of one Utah array channel, together; pairs on the same electrode are shown
as a variant without them.

In [ ]:
for t_ in ["V1V1", "V2V2"]:
    for label, d in [("all pairs", pairs[(pairs.type == t_) & pairs.sig10]),
                     ("different electrodes", pairs[(pairs.type == t_) & pairs.sig10 & (pairs.elec_i != pairs.elec_j)])]:
        l_ = d.lag10.values
        nz = l_[l_ != 0]
        pw = st.wilcoxon(nz).pvalue if len(nz) > 0 else np.nan
        pp = st.wilcoxon(l_, zero_method="pratt").pvalue
        pz = st.wilcoxon(l_, zero_method="zsplit").pvalue
        npos, nneg = (l_ > 0).sum(), (l_ < 0).sum()
        ps = st.binomtest(npos, npos + nneg).pvalue if npos + nneg > 0 else np.nan
        print(f"{t_} {label}: n = {len(l_)} of {(pairs.type == t_).sum()}, median {np.median(l_):.1f} ms, mean {l_.mean():.3f}, "
              f"zero {np.sum(l_ == 0)}, +{npos} / -{nneg}; Wilcoxon p = {pw:.3f} (zeros dropped), {pp:.3f} (Pratt), {pz:.3f} (zeros split); sign test p = {ps:.3f}")
fig, ax = plt.subplots(figsize=(5, 3))
for t_, c in [("V1V1", "C0"), ("V1V2", "0.5"), ("V2V2", "C3")]:
    l_ = pairs[(pairs.type == t_) & pairs.sig10].lag10
    ax.hist(l_, bins=np.arange(-10.5, 11), histtype="step", density=True, color=c, label=t_)
ax.set_xlabel("peak offset (ms)"); ax.legend(); ax.set_xlim(-10.5, 10.5)
print("all offsets plotted lie in -10..10 ms by construction")

**Result: not reproduced, resting on an assumption.** Both medians are 0.0 ms. For the 249
V2-V2 pairs neither test finds a difference from zero (Wilcoxon p = 0.861, sign test p =
0.703). For the 2620 V1-V1 pairs the Wilcoxon signed-rank test finds one (p = 0.016 with zeros
dropped, 0.027 and 0.028 with the other treatments of zeros) and the sign test does not (p =
0.076). The shift is small: mean 0.104 ms, 978 positive and 900 negative offsets. The outcome
takes the Wilcoxon signed-rank test as the paper's test (see the attempt record). Without pairs
on the same electrode, V2-V2 pairs also differ (Wilcoxon p = 0.030, sign test p = 0.033).

## figs3-efficacy

> For significant V1-V2 pairs with peak offsets above 1 ms, the median efficacy (CCG integral
> within 2.5 ms of the peak) is 0.39%, mean 0.69%. Expected 0.39 +/- 0.039 and 0.69 +/- 0.12.

Efficacy is the sum of the V1-rate-normalised CCG over the five 1 ms bins centred on the peak
(-2.5 to +2.5 ms). Readings: (a) the jitter-corrected CCG; (b) the raw CCG, shown both
without any correction and shuffle-corrected.

In [ ]:
e = sg[sg.lag10 > 1]
print(f"n = {len(e)} pairs with offset above 1 ms (of {len(sg)} significant)")
for nm, label in [("eff_jc", "(a) jitter-corrected"), ("eff_raw", "(b) raw, uncorrected"), ("eff_sc", "(b) raw, shuffle-corrected")]:
    print(f"{label}: median {e[nm].median():.3f}%, mean {e[nm].mean():.3f} +/- {e[nm].std() / np.sqrt(len(e)):.3f}% (SEM), range {e[nm].min():.3f} to {e[nm].max():.3f}")
print("per animal, (a):", e.groupby("animal").eff_jc.agg(["median", "mean", "count"]).round(3).to_dict("index"))
print("variant, all significant pairs, (a): median %.3f%%, mean %.3f%%" % (sg.eff_jc.median(), sg.eff_jc.mean()))

**Result: not reproduced.** For the 128 significant pairs with offset above 1 ms, the
jitter-corrected efficacy (reading a) has median 0.324% and mean 0.496 +/- 0.046%, below 0.39
+/- 0.039 and 0.69 +/- 0.12. On the raw CCG (reading b) the values are far above: median 6.494%
uncorrected and 0.820% shuffle-corrected. The median of reading a is 0.334, 0.559 and 0.258%
in the three animals.

## fig5a-rf-alignment and rf-sizes

Blocked, `metadata_insufficient`: RF centres and variances are in pixels, and the data set
gives no pixel-to-degree scale and no fixation point. The paper gives 1024 x 768 pixels at
110 cm without a screen size, so no scale follows from it. For information only, the values
in pixels: RF diameter as 4 times the square root of the mean variance (reading a) and the
ratio of V2 to V1 diameters, which needs no scale (the paper's values give 2.27 / 1.21 =
1.88); and the probability of a significant pair against the RF offset in pixels, first eye.

In [ ]:
def rf_of(rf, row):
    return rf[row, :, 0]
d1, d2, d1b, d2b = [], [], [], []
for k, v in sess_info.items():
    r1 = v["v1_rf"][:, :, 0]; r2 = v["v2_rf"][:, :, 0]
    ok1 = ~np.isnan(r1[:, 4]) & (r1[:, 4] > 0) & (r1[:, 5] > 0)
    ok2 = ~np.isnan(r2[:, 4]) & (r2[:, 4] > 0) & (r2[:, 5] > 0)
    d1 += list(4 * np.sqrt(r1[ok1, 4:6].mean(1))); d2 += list(4 * np.sqrt(r2[ok2, 4:6].mean(1)))
    d1b += list(4 * (r1[ok1, 4] * r1[ok1, 5]) ** 0.25); d2b += list(4 * (r2[ok2, 4] * r2[ok2, 5]) ** 0.25)
print(f"per channel with a fit: V1 n = {len(d1)}, mean diameter (a) {np.mean(d1):.1f} px, (b) {np.mean(d1b):.1f} px; "
      f"V2 n = {len(d2)}, (a) {np.mean(d2):.1f} px, (b) {np.mean(d2b):.1f} px")
print(f"ratio V2 / V1: (a) {np.mean(d2) / np.mean(d1):.2f}, (b) {np.mean(d2b) / np.mean(d1b):.2f}; the paper's degrees give {2.27 / 1.21:.2f}")
print(f"scale that would make the V1 mean 1.21 deg: {np.mean(d1) / 1.21:.1f} px/deg; the V2 mean 2.27 deg: {np.mean(d2) / 2.27:.1f} px/deg")

In [ ]:
offs = []
for k, v in sess_info.items():
    d = v12[v12.session == k]
    r1 = v["v1_rf"][:, :, 0]; r2 = v["v2_rf"][:, :, 0]
    c1 = v["v1_codes"][d.i.values, 0] - 1; c2 = v["v2_codes"][d.j.values, 0] - 1
    dx = r1[c1, 2] - r2[c2, 2]; dy = r1[c1, 3] - r2[c2, 3]
    offs.append(pd.DataFrame(dict(off=np.hypot(dx, dy), sig=d.sig10.values)))
offs = pd.concat(offs).dropna()
bins = np.arange(0, 201, 25)
g = offs.groupby(pd.cut(offs.off, bins))["sig"].agg(["mean", "sum", "count"])
print(f"pairs with both RFs fitted: {len(offs)} of {len(v12)}; RF offset beyond 200 px: {(offs.off > 200).sum()}")
print(g)

## Divergence analyses: groups

Middle-layer V2 neurons are the kept V2 units on electrodes whose site-average CCG has a
significant narrow peak (fig4c-narrow-peak-sites, 10 ms jitter). The V1 population is every
kept V1 unit of the session. Histograms of V1 states from all sessions are put on one support
of proportions (k / N1).

In [ ]:
mid_sites = set(map(tuple, sites.loc[sites.sig10, ["session", "electrode"]].values))
groups = {}
for k, d in div.items():
    groups[k] = np.array([(k, int(e)) in mid_sites for e in d["v2_elec"]])
n_mid = sum(g_.sum() for g_ in groups.values())
print("middle-layer V2 neurons per session:", {k: int(g_.sum()) for k, g_ in groups.items()}, " total", n_mid)
print("V1 population sizes:", {k: d["n_v1"] for k, d in div.items()})
epochs = sum(d["raw"][[18, 19], 0][:, groups[k]].sum() for k, d in div.items())
spikes = sum(d["n_spikes"][groups[k]].sum() for k, d in div.items())
print(f"V2 spikes of middle-layer neurons from 50 ms: {spikes}; 1 ms epochs (two per spike in reading e-a, minus those outside the window): {epochs}")
OFF = list(div[h.SESSIONS[0]]["offsets"])
WIN = h.JITTER_WINDOWS
N_BOOT = 5000

def offset_hist(groupmask, offs, which="raw", w=None, e=0):
    """(neurons, support) histogram pooled over the given offsets, spike (e=0) or control (e=1)."""
    idx = [OFF.index(o) for o in offs]
    if which == "raw":
        hs = [div[k]["raw"][idx, e][:, groupmask[k]].sum(0) for k in h.SESSIONS]
    else:
        hs = [div[k]["jit"][w][idx, e][:, groupmask[k]].sum(0) for k in h.SESSIONS]
    return h.to_global(hs, [div[k]["n_v1"] for k in h.SESSIONS])

def compare(raw_s, jit_s, raw_c, jit_c, rng, per_neuron=False, n_boot=N_BOOT):
    """Spike and control divergence, their bootstrap over neurons and the two p readings."""
    wts = h.boot_weights(raw_s.shape[0], n_boot, rng)
    ds = h.divergence(raw_s, jit_s, per_neuron=per_neuron); dc = h.divergence(raw_c, jit_c, per_neuron=per_neuron)
    bs = h.divergence(raw_s, jit_s, wts, per_neuron); bc = h.divergence(raw_c, jit_c, wts, per_neuron)
    return dict(spike=ds, control=dc, ratio=ds / dc, p_a=h.boot_p_paired(bs, bc), p_b=h.boot_p_indep(bs, bc),
                spike_lo=np.percentile(bs, 2.5), spike_hi=np.percentile(bs, 97.5),
                control_lo=np.percentile(bc, 2.5), control_hi=np.percentile(bc, 97.5), boot_s=bs, boot_c=bc)

def run_reading(groupmask, rng, before=True, reading="main"):
    """Spike against control divergence at the six windows under one reading of the shared method."""
    rows = []
    sign = 1 if before else -1
    o4 = [-3, -2, 2, 3]
    for w, win in enumerate(WIN):
        if reading in ("main", "d-b", "e-b2", "e-b3"):
            offs = {"e-b2": [2], "e-b3": [3]}.get(reading, [2, 3])
            offs = [sign * o for o in offs]
            rs, js = offset_hist(groupmask, offs, "raw", e=0), offset_hist(groupmask, offs, "jit", w, e=0)
            rc, jcn = offset_hist(groupmask, offs, "raw", e=1), offset_hist(groupmask, offs, "jit", w, e=1)
        else:
            ii = [o4.index(sign * o) for o in (2, 3)]
            key_r, key_j = {"s-b": ("raw_sb", "jit_sb"), "r-b": ("raw", "jit_rb")}[reading]
            def hh(e, jit):
                out = []
                for k in h.SESSIONS:
                    d = div[k]
                    if jit:
                        x = d[key_j][w][ii, e].sum(0)
                        if reading == "r-b":   # mean distribution of three realizations: add the main one
                            x = x + d["jit"][w][[OFF.index(sign * o) for o in (2, 3)], e].sum(0)
                    elif key_r == "raw":
                        x = d["raw"][[OFF.index(sign * o) for o in (2, 3)], e].sum(0)
                    else:
                        x = d[key_r][ii, e].sum(0)
                    out.append(x[groupmask[k]])
                return h.to_global(out, [div[k]["n_v1"] for k in h.SESSIONS])
            rs, js, rc, jcn = hh(0, False), hh(0, True), hh(1, False), hh(1, True)
        r = compare(rs, js, rc, jcn, rng, per_neuron=(reading == "d-b"))
        r.update(window=win, reading=reading)
        rows.append(r)
    return pd.DataFrame(rows)

rng = np.random.default_rng(6)
READINGS = ["main", "e-b2", "e-b3", "s-b", "r-b", "d-b"]
res6a = pd.concat([run_reading(groups, rng, True, r) for r in READINGS], ignore_index=True)
cols = ["reading", "window", "spike", "control", "ratio", "p_a", "p_b", "spike_lo", "spike_hi", "control_lo", "control_hi"]
print(res6a[cols].to_string(float_format=lambda x: f"{x:.3g}"))
for r in READINGS:
    d = res6a[res6a.reading == r]
    print(f"{r}: windows with spike > control at p < 0.05, p-a: {((d.p_a < 0.05) & (d.spike > d.control)).sum()} of 6, "
          f"p-b: {((d.p_b < 0.05) & (d.spike > d.control)).sum()} of 6")

In [ ]:
# per animal (main reading, point values; the bootstrap is over the pooled neurons)
for an in ["105", "106", "107"]:
    gm = {k: groups[k] & (k[:3] == an) for k in groups}
    if sum(x.sum() for x in gm.values()) == 0:
        print(an, "no middle-layer neurons"); continue
    vals = []
    for w, win in enumerate(WIN):
        rs, js = offset_hist(gm, [2, 3], "raw", e=0), offset_hist(gm, [2, 3], "jit", w, e=0)
        rc, jcn = offset_hist(gm, [2, 3], "raw", e=1), offset_hist(gm, [2, 3], "jit", w, e=1)
        vals.append(h.divergence(rs, js) / h.divergence(rc, jcn))
    print(f"animal {an}: {sum(x.sum() for x in gm.values())} neurons; spike / control divergence by window:", np.round(vals, 2))
fig, ax = plt.subplots(figsize=(5, 3))
d = res6a[res6a.reading == "main"]
ax.errorbar(d.window, d.spike, yerr=[d.spike - d.spike_lo, d.spike_hi - d.spike], color="C3", label="before V2 spike")
ax.errorbar(d.window, d.control, yerr=[d.control - d.control_lo, d.control_hi - d.control], color="C0", label="control")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("jitter window (ms)"); ax.set_ylabel("divergence (bits)"); ax.legend()
print("plotted range: %.2e to %.2e bits" % (min(d.spike_lo.min(), d.control_lo.min()), max(d.spike_hi.max(), d.control_hi.max())))

**Result: reproduced.** The 16 middle-layer sites hold 88 V2 neurons with 1,985,591 spikes from
50 ms. Under the main reading the divergence before V2 spikes is above control at all six
windows, 1.72 to 2.43 times, with p < 0.001 under both bootstrap readings (no draw of 5000
under p-a; p-b at most 0.00037). Every other reading (single 1 ms bins 2 or 3 ms before for e-b,
s-b, r-b, d-b) also gives 6 of 6 windows. In each animal the ratio is 1.50 or more at every
window.

## fig7a-threefold

> For jitter windows of 5 to 80 ms, divergence before middle-layer V2 spikes is roughly
> 3-fold that of control epochs. Expected ratio 3 +/- 1. Readings (a) geometric mean of the five
> ratios, (b) median, (c) at least four of five windows within tolerance.

In [ ]:
for r in READINGS:
    d = res6a[(res6a.reading == r) & (res6a.window <= 80)]
    rat = d.ratio.values
    print(f"{r}: ratios {np.round(rat, 3)}; (a) geometric mean {np.exp(np.log(rat).mean()):.3f}; (b) median {np.median(rat):.3f}; "
          f"(c) within 2 to 4: {np.sum((rat >= 2) & (rat <= 4))} of 5")

**Result: partially reproduced.** Divergence before V2 spikes is about twice the control
divergence, not three times. Under the main reading the ratios at 5 to 80 ms are 1.716 to 2.072,
geometric mean 1.906 and median 1.923, below the 2 to 4 tolerance, and 1 of 5 windows is in
range. The other readings give geometric means of 1.790 to 1.995. The single bin 3 ms before
gives a median of 2.011, at the edge of the tolerance; reading e-b, one bin centred 2.5 ms
before, cannot be formed from 1 ms data, and the bins at 2 and 3 ms bracket it (median 1.803 and
2.011).

## fig7a-above-shuffle

> Divergence both before middle-layer V2 spikes and in control epochs is significantly larger
> than divergence of shuffled data. Readings: trials permuted within orientation (a) or across
> all trials (b); epochs of the shuffled data spike-triggered (c) or control (d). Reproduced
> when the lower 95% bootstrap bound of both spike and control divergence is above the shuffled
> value at every window from 5 to 80 ms.

In [ ]:
nv1 = [div[k]["n_v1"] for k in h.SESSIONS]
d = res6a[(res6a.reading == "main") & (res6a.window <= 80)].reset_index(drop=True)
rows = []
for r, rl in enumerate(["a", "b"]):
    for e, el in enumerate(["c", "d"]):
        raw_ = h.to_global([div[k]["shuf"][r, e][groups[k]] for k in h.SESSIONS], nv1)
        for w in range(5):
            jit_ = h.to_global([div[k]["shuf_jit"][r, w, e][groups[k]] for k in h.SESSIONS], nv1)
            sh = h.divergence(raw_, jit_)
            rows.append(dict(shuffle=rl, epochs=el, window=WIN[w], shuffled=sh, spike_lo=d.spike_lo[w],
                             control_lo=d.control_lo[w], both_above=(d.spike_lo[w] > sh) and (d.control_lo[w] > sh)))
shuf = pd.DataFrame(rows)
print(shuf.to_string(float_format=lambda x: f"{x:.4g}"))
print(shuf.groupby(["shuffle", "epochs"]).both_above.sum().rename("windows of 5 where both bounds are above"))

**Result: reproduced.** At every window from 5 to 80 ms and under all four readings, the
shuffled divergence (1.82e-4 to 4.62e-4 bits) is below the lower 95% bound of both the spike
divergence (3.30e-4 to 1.36e-3) and the control divergence (2.01e-4 to 6.81e-4). The narrowest
margin is at 5 ms, control epochs of data shuffled within orientation: 2.006e-4 against a bound
of 2.013e-4, a margin within the spread of one shuffle realization.

## fig6b-not-after-v2

> In epochs 1.5 to 3.5 ms after middle-layer V2 spikes, V1 divergence does not differ from
> control epochs for jitter windows of 5 to 80 ms. Claim of absence, bootstrap readings p-a and
> p-b, p < 0.05.

In [ ]:
rng = np.random.default_rng(62)
res6b = pd.concat([run_reading(groups, rng, False, r) for r in READINGS], ignore_index=True)
print(res6b[cols].to_string(float_format=lambda x: f"{x:.3g}"))
for r in READINGS:
    d = res6b[(res6b.reading == r) & (res6b.window <= 80)]
    print(f"{r}: windows 5-80 ms with p < 0.05, p-a: {(d.p_a < 0.05).sum()}, p-b: {(d.p_b < 0.05).sum()}; "
          f"difference spike - control: {np.round((d.spike - d.control).values, 7)}")

**Result: not reproduced.** After V2 spikes the divergence is above control at 80 ms under both
bootstrap readings and every reading of the method (main: ratio 1.37, no draw of 5000 under p-a,
p-b = 0.0038; p-b 0.046 or less in the others), and at 40 ms under p-a (ratio 1.20). At 5 to
20 ms the ratios are 1.03 to 1.08 (main, p-a 0.10 to 0.48). The elevation after the spike is
smaller than before it (at 40 and 80 ms, 1.20 and 1.37 after against 1.96 and 2.07 before).

## fig6-slow-both-sides

> At a jitter window of 1280 ms, V1 divergence is elevated both before and after middle-layer
> V2 spikes.

In [ ]:
for label, res in [("before", res6a), ("after", res6b)]:
    d = res[res.window == 1280]
    for _, row in d.iterrows():
        print(f"{label} {row.reading}: spike {row.spike:.3g}, control {row.control:.3g}, ratio {row.ratio:.2f}, p-a {row.p_a:.4f}, p-b {row.p_b:.4f}")

**Result: reproduced.** At 1280 ms the divergence is above control both before (ratio 2.43) and
after V2 spikes (2.00), with p < 0.001 under both bootstrap readings in every reading of the
method.

## fig7d-time-course

> Enhanced V1 coordination is evident several milliseconds before middle-layer V2 spikes, but
> not in more distant epochs or after V2 spikes.

Difference in divergence (spike minus control) at offsets within 15 ms, windows 5 to 80 ms,
main reading otherwise. Readings (a) 2 ms epochs stepped by 1 ms (centre d + 0.5 for bins d
and d + 1); (b) single 1 ms bins (centre d). Positive centres are before the spike. Bootstrap
p (reading p-a, 2000 draws, a standard choice for this many comparisons), Bonferroni over the
offsets of a window.

In [ ]:
def time_course(groupmask, rng, n_boot=2000):
    rows = []
    for rd in ["a", "b"]:
        epochs_ = [([d, d + 1], d + 0.5) for d in range(-16, 16) if abs(d + 0.5) <= 15] if rd == "a" else [([d], float(d)) for d in range(-15, 16)]
        for w in range(5):
            for offs, c in epochs_:
                rs, js = offset_hist(groupmask, offs, "raw", e=0), offset_hist(groupmask, offs, "jit", w, e=0)
                rc, jcn = offset_hist(groupmask, offs, "raw", e=1), offset_hist(groupmask, offs, "jit", w, e=1)
                r = compare(rs, js, rc, jcn, rng, n_boot=n_boot)
                rows.append(dict(reading=rd, window=WIN[w], centre=c, diff=r["spike"] - r["control"], p=r["p_a"],
                                 p_b=r["p_b"], n_off=len(epochs_)))
    return pd.DataFrame(rows)

rng = np.random.default_rng(74)
tc = time_course(groups, rng)
def judge_tc(tc):
    out = []
    for (rd, win), d in tc.groupby(["reading", "window"]):
        best = d.loc[d["diff"].idxmax()]
        bad = d[d.sig_up & ((d.centre < 0) | (d.centre > 8))]
        out.append(dict(reading=rd, window=win, peak_centre=best.centre, peak_diff=best["diff"],
                        peak_in_1_6=1 <= best.centre <= 6, sig_before_1_6=int((d.sig_up & d.centre.between(1, 6)).sum()),
                        sig_after_or_far=len(bad), bad_centres=list(bad.centre)))
    return pd.DataFrame(out)
for pr in ["p", "p_b"]:
    tc["sig_up"] = (tc[pr] < 0.05 / tc.n_off) & (tc["diff"] > 0)
    jt = judge_tc(tc)
    print(f"bootstrap p reading {'p-a' if pr == 'p' else 'p-b'}:")
    print(jt.to_string(float_format=lambda x: f"{x:.3g}"))
print(tc.pivot_table(index="centre", columns=["reading", "window"], values="diff").loc[lambda x: x.index.isin([-10, -5, -2.5, -2, 0, 0.5, 1.5, 2, 2.5, 3, 3.5, 5, 10])].to_string(float_format=lambda x: f"{x:.1e}"))
fig, ax = plt.subplots(1, 5, figsize=(14, 2.6), sharey=True)
for i, win in enumerate(WIN[:5]):
    d = tc[(tc.reading == "a") & (tc.window == win)]
    ax[i].plot(d.centre, d["diff"], "k.-"); ax[i].axhline(0, color="0.7"); ax[i].set_title(f"{win} ms"); ax[i].set_xlabel("ms before V2 spike")
ax[0].set_ylabel("spike - control (bits)")
print("plotted range (reading a): %.2e to %.2e" % (tc[tc.reading == "a"]["diff"].min(), tc[tc.reading == "a"]["diff"].max()))

**Result: partially reproduced.** At every window from 5 to 80 ms the largest difference is 2.5
ms before the V2 spike for 2 ms epochs (reading a) and 3 ms before for 1 ms bins (reading b).
Some epochs after the spike or more than 8 ms before it are also above control after the
Bonferroni correction: with the paired bootstrap (p-a) at every window (2 to 20 epochs per
window), and with the independent one (p-b) at 80 ms only (6 epochs in reading a, 5 in reading
b). These differences are smaller than near the spike: at 80 ms, 2.7e-4 bits for the epoch 2.5
ms after against 8.3e-4 for the epoch 2.5 ms before.

## figs4-partner-not-needed

> V1 coordination is elevated before spikes of middle-layer V2 neurons both with and without a
> significant CCG peak with a recorded V1 neuron, slightly more for those with one.

In [ ]:
partner = set(map(tuple, v12.loc[v12.sig10, ["session", "j"]].values))
with_p = {k: groups[k] & np.array([(k, int(u)) in partner for u in div[k]["v2_units"]]) for k in groups}
without_p = {k: groups[k] & ~with_p[k] for k in groups}
rng = np.random.default_rng(84)
out = {}
for label, gm in [("with partner", with_p), ("without partner", without_p)]:
    n = sum(x.sum() for x in gm.values())
    r = run_reading(gm, rng, True, "main")
    r = r[r.window <= 80]
    out[label] = r
    print(f"{label}: {n} neurons; ratio {np.round(r.ratio.values, 2)}; p-a {np.round(r.p_a.values, 4)}; p-b {np.round(r.p_b.values, 4)}; "
          f"windows with spike > control at p < 0.05 (p-a): {((r.p_a < 0.05) & (r.spike > r.control)).sum()} of 5")
print("ratio larger with partner at", int((out["with partner"].ratio.values > out["without partner"].ratio.values).sum()), "of 5 windows")

**Result: reproduced.** 39 middle-layer neurons have a significant pair with a V1 unit and 49
have none. Both subsets have spike divergence above control at all five windows under p-a (no
draw of 5000), and under p-b at 5 of 5 with a partner and 4 of 5 without (at 5 ms p = 0.153). The
ratio is larger with a partner at all five windows (1.91 to 2.30 against 1.32 to 1.67).

## Blocked claims

- fig4-narrow-peak-depth, fig7b-nonmiddle-no-coordination, fig7e-nonmiddle-time-course,
  figs5-middle-layer-relay: need V2 depth or the order of sites in a penetration; the files
  hold one recording per V2 electrode per session and no depth (`data_missing`).
- fig7c-offset-rf-no-coordination: needs depth (500 to 1000 um from the layer 6/WM border)
  and RF offsets in degrees (`data_missing`).
- figs2a-v2-single-units, figs2b-v1-single-units: need unit SNR; the files hold no waveforms
  or SNR (`data_missing`).
- fig5a-rf-alignment, rf-sizes: need RF positions in degrees (`metadata_insufficient`).
- fig1-ffn-propagation, fig3-divergence-correlation, figs1-higher-order, fig8-architecture:
  simulations with no published code (`data: elsewhere`), not run in this attempt.

## Summary

| claim | outcome |
|---|---|
| fig4c-broad-ccg-peaks | reproduced |
| fig4c-narrow-peak-sites | not reproduced (0.516 of sites) |
| fig4-narrow-peak-depth | blocked, data_missing |
| fig5a-rf-alignment | blocked, metadata_insufficient |
| fig5-narrow-pairs-rare | not reproduced (0.0105 of pairs) |
| fig5b-v1v2-latency | partially reproduced (offset significant; median 2.0 ms, width 3.89 ms) |
| fig5b-intra-areal-zero-lag | not reproduced, assumed (V1-V1 Wilcoxon p = 0.016) |
| figs3-efficacy | not reproduced (median 0.324%, mean 0.496%) |
| fig6a-coordination-before-v2 | reproduced |
| fig7a-threefold | partially reproduced (ratio 1.91) |
| fig7a-above-shuffle | reproduced |
| fig6b-not-after-v2 | not reproduced (80 ms elevated after spikes) |
| fig6-slow-both-sides | reproduced |
| fig7d-time-course | partially reproduced |
| fig7b-nonmiddle-no-coordination, fig7e-nonmiddle-time-course, fig7c-offset-rf-no-coordination, figs5-middle-layer-relay | blocked, data_missing |
| figs2a-v2-single-units, figs2b-v1-single-units | blocked, data_missing |
| figs4-partner-not-needed | reproduced |
| rf-sizes | blocked, metadata_insufficient |
| fig1-ffn-propagation, fig3-divergence-correlation, figs1-higher-order, fig8-architecture | blocked, not_runnable |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- Spike times are at 1 ms in the files; the paper's CCGs used 0.1 ms bins. Peak offsets are
  whole milliseconds here.
- Units are kept by rate only; the SNR criterion cannot be applied.
- Sites are V2 electrodes in a session, 31 in all against 223; depth, penetration and layer
  are not known, so the non-middle-layer and offset-RF groups cannot be formed.
- Readings of open details follow the claims file: the CCG jitter window (10 ms; 25 ms as a
  variant), the epoch (two 1 ms bins; e-b approximated by one bin at 2 or 3 ms), the stratum of
  the jitter and control trials, one or three jitter realizations, pooled or per-neuron
  distributions, and paired or independent bootstrap p-values.

What was found about the data set:

- It holds 5 sessions from 3 animals, with 564 V1 and 147 V2 units; the descriptor's mean of
  112.2 V1 units per session does not match the 112.8 in the files.
- No V2 depth, penetration identity, unit SNR or waveforms; the paper's laminar and
  single-unit analyses cannot be repeated.
- RF positions and sizes are in pixels with no scale or fixation point; the paper does not
  give the screen size either.
- Narrow V1-V2 peaks are found at about half of the sites, against a fifth in the paper, so
  the deposited sessions are a selection the documentation does not describe.
- The descriptor's list of orientation codes does not hold for session 105l001p16, where code 7
  is 180 deg (sessions description); the population responses agree with the latter.

In [ ]:
print(f"notebook run time {time.time() - T0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")